In [35]:
#| default_exp _core

In [36]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [37]:
#| export 

from dataclasses import dataclass
from typing import Iterable, Sequence, Optional

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

import matplotlib.pyplot as plt

def to_device(batch, device):
    if isinstance(batch, torch.Tensor):
        return batch.to(device)
    if isinstance(batch, (list, tuple)):
        return type(batch)(to_device(x, device) for x in batch)
    if isinstance(batch, dict):
        return {k: to_device(v, device) for k, v in batch.items()}
    return batch


In [38]:
#| export

def make_tiny_dataloader(
    dataset: Dataset,
    n_items: int = 8,
    batch_size: int = 4,
    shuffle: bool = True,
) -> DataLoader:
    """
    Take the first n_items from `dataset` and build a tiny DataLoader
    for overfitting/debugging.
    """
    indices = list(range(min(n_items, len(dataset))))
    subset = torch.utils.data.Subset(dataset, indices)
    return DataLoader(subset, batch_size=batch_size, shuffle=shuffle, drop_last=True)


def batchify_video(video: torch.Tensor) -> torch.Tensor:
    """
    Ensure (b, c, t, h, w). If dataset returns (c, t, h, w), add batch dim.
    """
    if video.ndim == 4:  # (c, t, h, w)
        video = video.unsqueeze(0)
    assert video.ndim == 5, f"Expected 5D video tensor, got shape {video.shape}"
    return video

In [39]:
#| export

@dataclass
class TinyOverfitConfig:
    lr: float = 3e-4
    weight_decay: float = 0.0
    steps: int = 500
    log_every: int = 50
    mask_patches: bool = True
    max_grad_norm: Optional[float] = 1.0

def train_tiny_overfit(
    model: nn.Module,
    dataloader: DataLoader,
    cfg: TinyOverfitConfig,
    device: Optional[torch.device] = None,
):
    """
    Overfit the VideoTokenizer on a tiny dataset, logging recon / LPIPS loss.
    """
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.train()

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    step = 0
    losses = []

    while step < cfg.steps:
        for batch in dataloader:
            if isinstance(batch, (tuple, list)):
                video = batch[0]
            elif isinstance(batch, dict):
                # try some common keys
                video = batch.get("video") or batch.get("obs") or batch.get("image")
            else:
                video = batch

            video = batchify_video(video)
            video = to_device(video, device)

            opt.zero_grad(set_to_none=True)

            total_loss, tokenizer_losses = model(
                video,
                return_all_losses=True,
                mask_patches=cfg.mask_patches,
            )
            recon_loss, lpips_loss = tokenizer_losses.recon, tokenizer_losses.lpips  # ✅


            total_loss.backward()
            if cfg.max_grad_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
            opt.step()

            losses.append(
                {
                    "step": step,
                    "total": float(total_loss.detach().cpu()),
                    "recon": float(recon_loss.detach().cpu()),
                    "lpips": float(lpips_loss.detach().cpu()),
                }
            )

            if step % cfg.log_every == 0:
                print(
                    f"[{step:05d}] total={losses[-1]['total']:.4f} "
                    f"recon={losses[-1]['recon']:.4f} lpips={losses[-1]['lpips']:.4f}"
                )

            step += 1
            if step >= cfg.steps:
                break

    return losses


In [40]:
#| export

def plot_losses(losses: Sequence[dict], keys=("total", "recon", "lpips")):
    steps = [d["step"] for d in losses]
    for k in keys:
        vals = [d[k] for d in losses]
        plt.plot(steps, vals, label=k)
    plt.xlabel("step")
    plt.ylabel("loss")
    plt.legend()
    plt.title("VideoTokenizer tiny-overfit losses")
    plt.show()


In [66]:
#| export
from dreamer4.envs.pinpad import PinPad

# wrap the env to return float32 images between 0 and 1
class WrappedEnv:
    def __init__(self, env):
        self.env = env
        self.observation_space = env.observation_space
        self.action_space = env.action_space

    def reset(self):
        obs = self.env.reset()
        obs = obs / 255.0
        return obs

    def step(self, action):
        obs, reward, done, trunc, info = self.env.step(action)
        obs = obs / 255.0
        return obs, reward, done, trunc, info

import random
class MotionPlannerPinPad():
    def __init__(self, env):
        assert env.task == 'three'
        self.chosen_corner = random.choice([0,1,2])
        self.actions = []


    # move = [(0, 0), (0, 1), (0, -1), (1, 0), (-1, 0)][action]

    def refresh_actions(self):
        self.chosen_corner = random.choice([0,1,2])
        if self.chosen_corner == 0:
            actions = [1, 1, 3, 3, 1, 1, 3, 1, 3, 1, 3]
        elif self.chosen_corner == 1:
            actions = [2, 2, 4, 4, 2, 2, 4, 2, 4, 2, 4]
        elif self.chosen_corner == 2:
            actions = [2, 2, 3, 3, 2, 2, 2, 4, 4, 4, 4]
        else: raise ValueError
        self.actions = actions


    def sample(self):
        if not self.actions:
            self.refresh_actions()

        return self.actions.pop()

import cv2
def build_tiny_pinpad_dataset(device='cuda', n=1024, episode_length=8):
    env = PinPad('three', length=episode_length, extra_obs=False, size=[64, 64], random_starting_pos=True, device=device)
    # mp = MotionPlannerPinPad(env)
    env = WrappedEnv(env)
    obs = env.reset()
    # print the obs stats
    # make a tiny dataset
    eps_containing_success = 0; eps = 0
    ep_contained_success = False
    videos = []; curr_video = []
    for _ in range(n - 1):
        curr_video.append(obs)
        action = env.action_space.sample() # if random.random() > 0.5 else mp.sample()
        obs, reward, done, terminated, info = env.step(action)

        ep_contained_success = info['success'] or ep_contained_success
        if done or terminated:
            eps += 1
            eps_containing_success += 1 if ep_contained_success else 0
            ep_contained_success = False

            videos.append(torch.stack(curr_video))
            curr_video = []
            obs = env.reset()


    videos = torch.stack(videos)  # (n, t, c, h, w)
    videos = videos.permute(0, 2, 1, 3, 4)  # (n, c, t, h, w)

    print(f"Built tiny PinPad dataset with {videos.shape[0]} videos of shape {videos.shape[1:]} - (c, t, h, w). Success rate {eps_containing_success / eps:1.1%}")
    return torch.utils.data.TensorDataset(videos), env
    # return videos

In [67]:
videos, env = build_tiny_pinpad_dataset(episode_length=100)

pads {'2', '1', '3'} target ('1', '2', '3')

!!!!SUCCESS!!!
Built tiny PinPad dataset with 10 videos of shape torch.Size([3, 100, 64, 64]) - (c, t, h, w). Success rate 10.0%


In [68]:
#| hide
import nbdev; nbdev.nbdev_export()